# Buscador y Resumidor de Papers Academicos - arXiv

Este notebook permite:
- Buscar papers en **arXiv** por palabras clave, autor o categoria
- Ver el **resumen automatico** de cada paper
- **Descargar** los resultados en CSV o PDF directamente al PC

---

## Paso 1: Instalar dependencias

In [ ]:
!pip install arxiv requests pandas sumy nltk fpdf2 --quiet
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
print('Dependencias instaladas correctamente.')

## Paso 2: Importar librerias

In [ ]:
import arxiv
import pandas as pd
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer
from sumy.nlp.stemmers import Stemmer
from sumy.utils import get_stop_words
from fpdf import FPDF
from google.colab import files
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import textwrap
import datetime
import re

print('Librerias importadas correctamente.')

## Paso 3: Funciones principales

In [ ]:
def resumir_texto(texto, oraciones=3, idioma='english'):
    """Genera un resumen extractivo del texto usando LSA."""
    try:
        parser = PlaintextParser.from_string(texto, Tokenizer(idioma))
        stemmer = Stemmer(idioma)
        summarizer = LsaSummarizer(stemmer)
        summarizer.stop_words = get_stop_words(idioma)
        resumen = summarizer(parser.document, oraciones)
        return ' '.join(str(s) for s in resumen)
    except Exception:
        # Si el texto es corto, devolver el texto directamente
        return texto[:600] + '...' if len(texto) > 600 else texto


def buscar_papers(query, max_results=10, sort_by='relevance', categoria=None):
    """Busca papers en arXiv y devuelve lista de resultados."""
    sort_map = {
        'relevance': arxiv.SortCriterion.Relevance,
        'fecha': arxiv.SortCriterion.SubmittedDate,
        'citas': arxiv.SortCriterion.Relevance
    }

    if categoria:
        query = f'cat:{categoria} AND ({query})'

    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=sort_map.get(sort_by, arxiv.SortCriterion.Relevance),
        sort_order=arxiv.SortOrder.Descending
    )

    resultados = []
    client = arxiv.Client()
    for paper in client.results(search):
        abstract = paper.summary.replace('\n', ' ')
        resumen = resumir_texto(abstract, oraciones=3)
        resultados.append({
            'titulo': paper.title,
            'autores': ', '.join(a.name for a in paper.authors),
            'fecha': paper.published.strftime('%Y-%m-%d') if paper.published else 'N/A',
            'categorias': ', '.join(paper.categories),
            'abstract': abstract,
            'resumen': resumen,
            'url': paper.entry_id,
            'pdf_url': paper.pdf_url,
            'doi': paper.doi or 'N/A'
        })
    return resultados


def mostrar_resultados_html(resultados):
    """Muestra los resultados en formato HTML dentro del notebook."""
    if not resultados:
        display(HTML('<p style="color:red">No se encontraron resultados.</p>'))
        return

    html = f'<h3>Se encontraron {len(resultados)} papers:</h3>'
    for i, p in enumerate(resultados, 1):
        html += f'''
        <div style="border:1px solid #ddd; border-radius:8px; padding:16px; margin:12px 0;
                    background:#f9f9f9; font-family:Arial, sans-serif;">
          <h4 style="margin:0 0 6px 0; color:#1a0dab;">#{i} — {p['titulo']}</h4>
          <p style="margin:2px 0; font-size:13px; color:#555;">
            <b>Autores:</b> {p['autores']}</p>
          <p style="margin:2px 0; font-size:13px; color:#555;">
            <b>Fecha:</b> {p['fecha']} &nbsp;|&nbsp;
            <b>Categorias:</b> {p['categorias']}</p>
          <p style="margin:2px 0; font-size:13px; color:#555;">
            <b>DOI:</b> {p['doi']}</p>
          <details style="margin-top:8px;">
            <summary style="cursor:pointer; color:#1a0dab; font-size:13px;">Ver abstract completo</summary>
            <p style="font-size:12px; color:#333; margin-top:6px;">{p['abstract']}</p>
          </details>
          <div style="background:#eef6ff; border-left:4px solid #1a73e8;
                      padding:10px; margin-top:10px; border-radius:4px;">
            <b style="font-size:13px;">Resumen automatico:</b>
            <p style="font-size:13px; margin:4px 0;">{p['resumen']}</p>
          </div>
          <p style="margin-top:8px; font-size:13px;">
            <a href="{p['url']}" target="_blank">Ver en arXiv</a> &nbsp;|&nbsp;
            <a href="{p['pdf_url']}" target="_blank">Descargar PDF</a>
          </p>
        </div>
        '''
    display(HTML(html))


def exportar_csv(resultados, nombre_archivo=None):
    """Exporta los resultados a un CSV y lo descarga."""
    if not resultados:
        print('No hay resultados para exportar.')
        return
    if not nombre_archivo:
        ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        nombre_archivo = f'papers_arxiv_{ts}.csv'
    df = pd.DataFrame(resultados)
    df.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')
    print(f'Archivo generado: {nombre_archivo}')
    files.download(nombre_archivo)


def exportar_pdf(resultados, nombre_archivo=None):
    """Exporta los resultados a un PDF y lo descarga."""
    if not resultados:
        print('No hay resultados para exportar.')
        return
    if not nombre_archivo:
        ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        nombre_archivo = f'papers_arxiv_{ts}.pdf'

    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()
    pdf.set_font('Helvetica', 'B', 16)
    pdf.cell(0, 10, 'Resultados de busqueda - arXiv', ln=True, align='C')
    pdf.set_font('Helvetica', '', 10)
    pdf.cell(0, 8, f'Generado: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}', ln=True, align='C')
    pdf.ln(6)

    def safe_text(text):
        return text.encode('latin-1', 'replace').decode('latin-1')

    for i, p in enumerate(resultados, 1):
        pdf.set_font('Helvetica', 'B', 12)
        pdf.multi_cell(0, 7, safe_text(f'{i}. {p["titulo"]}'))
        pdf.set_font('Helvetica', '', 9)
        pdf.multi_cell(0, 6, safe_text(f'Autores: {p["autores"]}'))
        pdf.multi_cell(0, 6, safe_text(f'Fecha: {p["fecha"]}  |  Categorias: {p["categorias"]}'))
        pdf.multi_cell(0, 6, safe_text(f'DOI: {p["doi"]}'))
        pdf.multi_cell(0, 6, safe_text(f'URL: {p["url"]}'))
        pdf.ln(2)
        pdf.set_font('Helvetica', 'B', 9)
        pdf.cell(0, 6, 'Resumen automatico:', ln=True)
        pdf.set_font('Helvetica', '', 9)
        pdf.multi_cell(0, 6, safe_text(p['resumen']))
        pdf.set_font('Helvetica', 'B', 9)
        pdf.cell(0, 6, 'Abstract completo:', ln=True)
        pdf.set_font('Helvetica', '', 8)
        pdf.multi_cell(0, 5, safe_text(p['abstract']))
        pdf.ln(4)
        pdf.set_draw_color(180, 180, 180)
        pdf.line(10, pdf.get_y(), 200, pdf.get_y())
        pdf.ln(4)

    pdf.output(nombre_archivo)
    print(f'Archivo generado: {nombre_archivo}')
    files.download(nombre_archivo)


print('Funciones cargadas correctamente.')

## Paso 4: Interfaz de busqueda interactiva

Ejecuta la celda y usa los controles para buscar papers.

In [ ]:
# ─── Widgets ───────────────────────────────────────────────────────────────
CATEGORIAS = [
    ('Todas las categorias', ''),
    ('Inteligencia Artificial (cs.AI)', 'cs.AI'),
    ('Machine Learning (cs.LG)', 'cs.LG'),
    ('Procesamiento de Lenguaje Natural (cs.CL)', 'cs.CL'),
    ('Vision por Computadora (cs.CV)', 'cs.CV'),
    ('Robotica (cs.RO)', 'cs.RO'),
    ('Fisica (physics)', 'physics'),
    ('Matematicas (math)', 'math'),
    ('Biologia (q-bio)', 'q-bio'),
    ('Economia (econ)', 'econ'),
    ('Estadistica (stat)', 'stat'),
]

w_query = widgets.Text(
    value='',
    placeholder='Ej: large language models, deep learning, reinforcement learning...',
    description='Busqueda:',
    layout=widgets.Layout(width='70%'),
    style={'description_width': '80px'}
)

w_categoria = widgets.Dropdown(
    options=[(label, val) for label, val in CATEGORIAS],
    value='',
    description='Categoria:',
    layout=widgets.Layout(width='50%'),
    style={'description_width': '80px'}
)

w_max = widgets.IntSlider(
    value=10,
    min=1,
    max=50,
    step=1,
    description='Cantidad:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='50%')
)

w_sort = widgets.RadioButtons(
    options=[('Relevancia', 'relevance'), ('Mas recientes', 'fecha')],
    value='relevance',
    description='Ordenar:',
    style={'description_width': '80px'}
)

w_oraciones = widgets.IntSlider(
    value=3,
    min=1,
    max=6,
    step=1,
    description='Oraciones resumen:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='50%')
)

btn_buscar = widgets.Button(
    description='Buscar papers',
    button_style='primary',
    icon='search',
    layout=widgets.Layout(width='160px', height='36px')
)

btn_csv = widgets.Button(
    description='Descargar CSV',
    button_style='success',
    icon='download',
    layout=widgets.Layout(width='160px', height='36px'),
    disabled=True
)

btn_pdf = widgets.Button(
    description='Descargar PDF',
    button_style='warning',
    icon='file-pdf-o',
    layout=widgets.Layout(width='160px', height='36px'),
    disabled=True
)

out_status = widgets.Output()
out_results = widgets.Output()

resultados_globales = []

# ─── Logica de botones ─────────────────────────────────────────────────────
def on_buscar(b):
    global resultados_globales
    query = w_query.value.strip()
    if not query:
        with out_status:
            clear_output()
            print('Por favor escribe una palabra clave para buscar.')
        return

    btn_buscar.disabled = True
    btn_csv.disabled = True
    btn_pdf.disabled = True

    with out_status:
        clear_output()
        print(f'Buscando "{query}" en arXiv... espera un momento.')

    with out_results:
        clear_output()

    try:
        resultados_globales = buscar_papers(
            query=query,
            max_results=w_max.value,
            sort_by=w_sort.value,
            categoria=w_categoria.value or None
        )
        with out_status:
            clear_output()
            print(f'Listo. {len(resultados_globales)} papers encontrados.')
        with out_results:
            mostrar_resultados_html(resultados_globales)
        btn_csv.disabled = False
        btn_pdf.disabled = False
    except Exception as e:
        with out_status:
            clear_output()
            print(f'Error al buscar: {e}')
    finally:
        btn_buscar.disabled = False


def on_csv(b):
    with out_status:
        clear_output()
        print('Generando CSV...')
    exportar_csv(resultados_globales)


def on_pdf(b):
    with out_status:
        clear_output()
        print('Generando PDF...')
    exportar_pdf(resultados_globales)


btn_buscar.on_click(on_buscar)
btn_csv.on_click(on_csv)
btn_pdf.on_click(on_pdf)

# ─── Layout ────────────────────────────────────────────────────────────────
display(HTML('<h3 style="font-family:Arial">Buscador de Papers - arXiv</h3>'))
display(widgets.VBox([
    w_query,
    widgets.HBox([w_categoria, w_sort]),
    widgets.HBox([w_max, w_oraciones]),
    widgets.HBox([btn_buscar, btn_csv, btn_pdf]),
    out_status,
    out_results
]))

## Paso 5 (opcional): Busqueda rapida por codigo

Si prefieres no usar la interfaz grafica, puedes ejecutar directamente:

In [ ]:
# ── Configura tu busqueda aqui ──────────────────────────────────────────────
QUERY      = 'retrieval augmented generation'   # Terminos de busqueda
MAX        = 5                                  # Numero de papers (1-50)
SORT       = 'fecha'                            # 'relevance' o 'fecha'
CATEGORIA  = 'cs.CL'                            # Dejar '' para todas
# ────────────────────────────────────────────────────────────────────────────

resultados = buscar_papers(QUERY, max_results=MAX, sort_by=SORT, categoria=CATEGORIA)
mostrar_resultados_html(resultados)

# Descomentar para exportar:
# exportar_csv(resultados)
# exportar_pdf(resultados)

---

## Licencias, Politicas de Datos y Uso de la API

### Quien opera arXiv
arXiv es operado por la **Universidad de Cornell** con apoyo de la Simons Foundation y afiliados academicos. Es un repositorio de preprints sin fines de lucro fundado en 1991, reconocido mundialmente en ciencias exactas e ingenieria.

---

### Licencia de los datos

| Tipo de dato | Licencia |
|---|---|
| Metadatos (titulo, autores, abstract, categorias, fecha) | **Dominio publico** — uso libre sin restricciones |
| Libreria `arxiv` (cliente Python) | **Apache License 2.0** — uso libre incluyendo comercial |
| Contenido de cada paper (PDF) | Licencia propia del autor (ver tabla abajo) |

Las licencias mas comunes que los autores asignan a sus papers en arXiv son:

| Licencia | Que permite |
|---|---|
| **CC BY 4.0** | Usar, copiar, distribuir y adaptar con atribucion al autor |
| **CC BY-SA 4.0** | Igual que CC BY pero obras derivadas deben usar la misma licencia |
| **CC BY-NC-SA 4.0** | Solo uso **no comercial**, obras derivadas con misma licencia |
| **CC BY-NC-ND 4.0** | Solo lectura, **no comercial**, sin modificaciones |
| **Licencia arXiv** | Descarga personal permitida; redistribucion y modificacion **no permitidas** |

> Verifica la licencia de cada paper en su pagina de arXiv antes de reutilizar su contenido en publicaciones o productos.

---

### Terminos de uso de la API

- **Gratuita y sin registro** obligatorio
- **Limite recomendado:** maximo 1 peticion cada 3 segundos para no sobrecargar los servidores
- **Permitido:** busqueda de metadatos, lectura de abstracts, descarga individual de PDFs para investigacion
- **No permitido:** scraping masivo automatizado de PDFs, redistribucion de contenido sin respetar licencias individuales
- Documentacion oficial: https://info.arxiv.org/help/api/index.html

---

### Privacidad y seguridad de datos

- Este notebook **no recopila ni almacena** ningun dato personal del usuario
- Las consultas se envian directamente desde tu sesion (Colab/Kaggle) a los servidores de arXiv
- arXiv puede registrar direcciones IP que usan su API con fines de monitoreo de uso
- Los archivos CSV y PDF se generan y descargan localmente; **no se envian a servidores externos**

---

### Nota de citacion academica

Si utilizas papers encontrados con este notebook en tu investigacion o publicaciones, **cita cada paper individualmente** usando su DOI o URL de arXiv. No se requiere citar este notebook salvo que tu institucion lo exija expresamente.